#### simple genai app using langchain and openai

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [3]:
from langchain_community.document_loaders import WebBaseLoader

c:\Users\dell\OneDrive\Documents\Desktop\Langchain\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader = WebBaseLoader("https://www.ibm.com/think/topics/artificial-intelligence")
data = loader.load()
print(data)

[Document(metadata={'source': 'https://www.ibm.com/think/topics/artificial-intelligence', 'title': 'What Is Artificial Intelligence (AI)? | IBM', 'description': 'Artificial intelligence (AI) is technology that enables computers and machines to simulate human learning, comprehension, problem solving, decision-making, creativity and autonomy.', 'language': 'en'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\nWhat Is Artificial Intelligence (AI)? | IBM\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                        \n                        \n\n\n  \n  \n    \n    What is artificial intelligence (AI)?\n\n  \n\n\n\n\n\n    \n\n\n                    \n\n\n\n\n\n    By \n    \n        Cole Stryker\n        , \n    \n         Eda Kavlakoglu\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n        What

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(data)

In [7]:
from langchain_openai import OpenAIEmbeddings

In [8]:
embeddings = OpenAIEmbeddings()

In [9]:
from langchain_community.vectorstores import FAISS

In [ ]:
vector_store_db = FAISS.from_documents(documents, embeddings)

In [ ]:
vector_store_db

In [ ]:
query = "What is Artificial Intelligence ?"
result = vector_store_db.similarity_search(query)
print(result[0].page_content)

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based on the provided context.
    <context>
    {context}
    </context>
"""
)

documents_chain = StuffDocumentsChain(
    llm=embeddings,
    prompt=prompt
)

In [ ]:
from langchain_core.documents import Document
documents_chain.invoke({
    "input":"What is Artificial Intelligence ?",
    "context":[Document(page_content=result[0].page_content)]
})

In [ ]:
retriever = vector_store_db.as_retriever()
from langchain.chains import create_retriever_chains
retriever_chain = create_retriever_chains(
    retriever=retriever,
    combine_documents_chain=documents_chain
)

In [ ]:
retriever_chain

In [ ]:
returner = retriever_chain.invoke({"input": "What is Artificial Intelligence ?"})
print(returner)